# Run ABG-HKG-AOG v7 Full

This notebook runs the full cache-runnable v7 path: fixed v6 build, v7 block-pursuit bank, recurrent re-query, native part-OR diagnostics, port relations, and visibility ledger.


In [ ]:
from pathlib import Path
import subprocess
REPO_DIR = Path('/home/dfli/instance_slot_aog/clean_v18_v39_v42')
TRAIN_CACHE = REPO_DIR / 'artifacts/strict_aog_v6/train_strict_aog_terminals.pt'
VAL_CACHE = REPO_DIR / 'artifacts/strict_aog_v6/val_strict_aog_terminals.pt'
ART_DIR = REPO_DIR / 'artifacts/abg_hkg_aog_v7_full'
BUNDLE = ART_DIR / 'abg_hkg_aog_v7_full_bundle.pt'
PART_TEMPLATE_BANK = ART_DIR / 'part_template_bank.pt'
BLOCK_BANK = ART_DIR / 'block_pursuit_bank.pt'
RUN_DIR = REPO_DIR / 'runs/abg_hkg_aog_v7_full'
BEST_CKPT = RUN_DIR / 'checkpoints/strict_aog_best.pt'
def run(cmd):
    cmd = [str(x) for x in cmd]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('repo:', REPO_DIR)
print('train cache:', TRAIN_CACHE)
print('val cache:', VAL_CACHE)

## 1. Pull branch and install


In [ ]:
run(['git','fetch','origin','pra-aog-v6-gpu-terminal-cache'])
run(['git','switch','pra-aog-v6-gpu-terminal-cache'])
run(['git','pull','--ff-only','origin','pra-aog-v6-gpu-terminal-cache'])
run(['python','-m','pip','install','-e','.[dev,vision]'])

## 2. Tests


In [ ]:
run(['pytest','-q','tests/test_abg_hkg_aog_v7_full.py','tests/test_abg_hkg_aog_v7.py','tests/test_pra_aog_v6_template_hierarchy.py','tests/test_hier_pra_aog.py'])

## 3. Build full v7 artifacts


In [ ]:
ART_DIR.mkdir(parents=True, exist_ok=True)
run(['python','scripts/build_abg_hkg_aog_v7_full.py','--cache',TRAIN_CACHE,'--out',BUNDLE,'--part-template-out',PART_TEMPLATE_BANK,'--block-bank-out',BLOCK_BANK,'--num-templates-per-class','5','--max-slots-per-template','14','--max-slots-per-part','4','--part-template-grid-size','3','--part-template-min-support','6','--part-template-max-per-part','6','--block-max-blocks','32'])

## 4. Smoke train


In [ ]:
run(['python','scripts/run_abg_hkg_aog_v7_full.py','--bundle',BUNDLE,'--part-template-bank',PART_TEMPLATE_BANK,'--train-cache',TRAIN_CACHE,'--val-cache',VAL_CACHE,'--save-dir',str(RUN_DIR)+'_smoke','--device','auto','--batch-size','4','--epochs','1','--v7-max-rounds','1','--v7-max-queries','2','--part-or-score-weight','0.20','--max-train-batches','2','--max-val-batches','2','--num-workers','0'])

## 5. Full train


In [ ]:
run(['python','scripts/run_abg_hkg_aog_v7_full.py','--bundle',BUNDLE,'--part-template-bank',PART_TEMPLATE_BANK,'--train-cache',TRAIN_CACHE,'--val-cache',VAL_CACHE,'--save-dir',RUN_DIR,'--device','auto','--batch-size','16','--epochs','20','--v7-max-rounds','1','--v7-max-queries','2','--part-or-score-weight','0.20','--preload-cache','--num-workers','0'])

## 6. Inference summary


In [ ]:
run(['python','scripts/infer_abg_hkg_aog_v7_full.py','--bundle',BUNDLE,'--part-template-bank',PART_TEMPLATE_BANK,'--cache',VAL_CACHE,'--checkpoint',BEST_CKPT,'--out-dir',RUN_DIR/'inference','--sample-index','0','--device','auto','--v7-max-rounds','1','--v7-max-queries','2'])